# P105 — Product Cannibalization & Demand Shift Analysis
## Phase 4: Product Performance Analysis

**Objective:** Identify best/worst performing products, compare categories and regions, and analyze revenue contributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
os.makedirs('../Images/Phase4', exist_ok=True)

LAUNCH_DATE = pd.Timestamp('2024-06-01')
LAUNCHED_PRODUCTS = ['P1', 'P2', 'P3', 'P4', 'P5']

## Load Cleaned Data

In [ ]:
df = pd.read_csv("../Data/cannibalization_cleaned.csv")
df['Date'] = pd.to_datetime(df['Date'])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1. Top & Bottom Products by Sales

In [ ]:
product_sales = df.groupby('Product_ID').agg(
    Total_Sales   = ('Sales', 'sum'),
    Avg_Sales     = ('Sales', 'mean'),
    Total_Revenue = ('Revenue', 'sum')
).round(2).sort_values('Total_Sales', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Top 10
top10 = product_sales.head(10)
colors = ['red' if p in LAUNCHED_PRODUCTS else 'steelblue' for p in top10.index]
ax1.barh(top10.index, top10['Total_Sales'], color=colors)
ax1.set_title('Top 10 Products by Total Sales', fontweight='bold')
ax1.set_xlabel('Total Sales')
ax1.invert_yaxis()

# Bottom 10
bot10 = product_sales.tail(10)
colors = ['red' if p in LAUNCHED_PRODUCTS else 'steelblue' for p in bot10.index]
ax2.barh(bot10.index, bot10['Total_Sales'], color=colors)
ax2.set_title('Bottom 10 Products by Total Sales', fontweight='bold')
ax2.set_xlabel('Total Sales')
ax2.invert_yaxis()

plt.suptitle('Product Rankings (Red = Launched Product)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/Phase4/01_top_bottom_products.png', bbox_inches='tight')
plt.show()

print("Full Product Ranking:")
print(product_sales.to_string())

## 2. Product Revenue Comparison

In [ ]:
product_rev = df.groupby('Product_ID')['Revenue'].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors = ['red' if p in LAUNCHED_PRODUCTS else 'steelblue' for p in product_rev.index]
plt.barh(product_rev.index, product_rev.values, color=colors)
plt.title('Total Revenue by Product (Red = Launched)', fontweight='bold')
plt.xlabel('Total Revenue')
plt.tight_layout()
plt.savefig('../Images/Phase4/02_revenue_by_product.png', bbox_inches='tight')
plt.show()

## 3. Category Performance Comparison

In [ ]:
cat_stats = df.groupby('Category').agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Revenue = ('Revenue', 'sum'),
    Avg_Price     = ('Price', 'mean'),
    Avg_Rating    = ('Rating', 'mean'),
    Products      = ('Product_ID', 'nunique')
).round(2).sort_values('Total_Revenue', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cat_stats['Total_Sales'].plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'))
axes[0].set_title('Total Sales', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

cat_stats['Total_Revenue'].plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2'))
axes[1].set_title('Total Revenue', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

cat_stats['Avg_Rating'].plot(kind='bar', ax=axes[2], color=sns.color_palette('Set2'))
axes[2].set_title('Avg Rating', fontweight='bold')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Category Performance', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/Phase4/03_category_performance.png', bbox_inches='tight')
plt.show()

print(cat_stats.to_string())

## 4. Region × Product Performance Heatmap

In [ ]:
region_prod = df.groupby(['Region', 'Product_ID'])['Sales'].sum().unstack(fill_value=0)

plt.figure(figsize=(14, 5))
sns.heatmap(region_prod, cmap='YlOrRd', annot=False, linewidths=0.5)
plt.title('Sales Heatmap — Region × Product', fontweight='bold')
plt.tight_layout()
plt.savefig('../Images/Phase4/04_region_product_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Product Growth Analysis (Before vs After Launch)

In [ ]:
growth = df.groupby(['Product_ID', 'Period_Flag'])['Sales'].mean().unstack()
growth.columns = ['After', 'Before']
growth['Growth_Pct'] = ((growth['After'] - growth['Before']) / growth['Before'] * 100).round(2)
growth.sort_values('Growth_Pct', ascending=True, inplace=True)

plt.figure(figsize=(10, 8))
colors = ['green' if x > 0 else 'red' for x in growth['Growth_Pct']]
plt.barh(growth.index, growth['Growth_Pct'], color=colors)
plt.axvline(0, color='gray', linewidth=1)
plt.title('Sales Growth % — After vs Before Launch', fontweight='bold')
plt.xlabel('Growth %')
plt.tight_layout()
plt.savefig('../Images/Phase4/05_product_growth.png', bbox_inches='tight')
plt.show()

print("Products with Negative Growth (Potential Cannibalization):")
print(growth[growth['Growth_Pct'] < 0][['Before', 'After', 'Growth_Pct']].to_string())

## 6. Revenue Contribution by Product

In [ ]:
rev_share = df.groupby('Product_ID')['Revenue'].sum()
rev_share_pct = (rev_share / rev_share.sum() * 100).round(2).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(rev_share_pct.index, rev_share_pct.values, color=sns.color_palette('husl', len(rev_share_pct)))
plt.title('Revenue Contribution (%) by Product', fontweight='bold')
plt.ylabel('Revenue Share (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../Images/Phase4/06_revenue_contribution.png', bbox_inches='tight')
plt.show()

print(rev_share_pct.to_string())

## 7. Product Group Performance — Affected vs Unaffected

In [ ]:
group_stats = df.groupby('Product_Group').agg(
    Total_Sales   = ('Sales', 'sum'),
    Avg_Sales     = ('Sales', 'mean'),
    Total_Revenue = ('Revenue', 'sum'),
    Products      = ('Product_ID', 'nunique')
).round(2)
group_stats['Is_Affected'] = group_stats.index.isin(['G1', 'G2'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['red' if g in ['G1', 'G2'] else 'steelblue' for g in group_stats.index]
ax1.bar(group_stats.index, group_stats['Total_Sales'], color=colors)
ax1.set_title('Total Sales by Group (Red = Affected)', fontweight='bold')
ax1.set_ylabel('Total Sales')

ax2.bar(group_stats.index, group_stats['Total_Revenue'], color=colors)
ax2.set_title('Total Revenue by Group (Red = Affected)', fontweight='bold')
ax2.set_ylabel('Total Revenue')

plt.tight_layout()
plt.savefig('../Images/Phase4/07_group_comparison.png', bbox_inches='tight')
plt.show()

print(group_stats.to_string())

## Phase 4 Summary

**Key Findings:**
- Products vary significantly in revenue contribution (high-price items dominate revenue)
- P6 shows negative growth after launch — confirming cannibalization
- Groups G1 and G2 (affected groups) show different dynamics vs unaffected groups
- Household and Personal Care categories have highest revenue due to higher prices
- Regional performance is relatively balanced

**Next:** Phase 5 — Cannibalization Deep Dive